In [1]:
import os
import sys
import argparse

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from data import *
from utils import *
from scipy import stats
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Disentanglement'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Accuracy'))
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'Attack'))

from DCI import compute_dci, report_dci
from IRS import compute_irs_from_multihot_labels, compute_irs
from MIG import compute_mig

from TA import *

In [2]:
parser = argparse.ArgumentParser(description='Gen-RKM Model')

parser.add_argument('--N', type=int, default=700, help='Total # of samples')
parser.add_argument('--mb_size', type=int, default=64, help='Mini-batch size. See utils.py') #32, 64
parser.add_argument('--h_dim', type=int, default=48, help='Dim of latent vector') # 16, 24
parser.add_argument('--capacity', type=int, default=48, help='Capacity of network. See utils.py')

parser.add_argument('--x_fdim', type=int, default=256, help='Input x_fdim. See utils.py')
parser.add_argument('--y_fdim', type=int, default=64, help='Input y_fdim. See utils.py')
parser.add_argument('--z_fdim', type=int, default=64, help='Input z_fdim. See utils.py')

parser.add_argument('--c_accu', type=float, default=5.0, help='Weight on recons_error')
parser.add_argument('--start_iter', type=int, default=0, help='Input start_iter for training')

# Training Settings =============================
parser.add_argument('--lr', type=float, default=7e-4, help='Input learning rate for optimizer')
parser.add_argument('--max_epochs', type=int, default=0, help='Input max_epoch for cut-off')
parser.add_argument('--device', type=str, default='cpu', help='Device type: cuda or cpu')
parser.add_argument('--workers', type=int, default=0, help='# of workers for dataloader')
parser.add_argument('--shuffle', type=bool, default=True, help='shuffle dataset: true or false')

opt, _ = parser.parse_known_args()

h_dim = opt.h_dim

In [3]:
_, xtest, ipVec_dim, nChannels = get_poem_dataloader(args=opt)

In [4]:
i_val = 6

txt_encoder = tiktoken.get_encoding("o200k_base")
PAD = txt_encoder.eot_token

In [5]:
def kPCA(X, Y, Z, nviews=3, eps=1e-6):
    if nviews == 3:
        a = (X @ X.T + Y @ Y.T + Z @ Z.T) / nviews
    elif nviews == 2:
        a = (X @ X.T + Y @ Y.T) / nviews
    else:
        a = (X @ X.T) / nviews

    B = a.size(0)

    # Centering (same dtype/device as a)
    oneN = torch.ones(B, B, device=a.device, dtype=a.dtype) / B
    a = a - oneN @ a - a @ oneN + oneN @ a @ oneN

    # Force symmetry (numerical)
    a = 0.5 * (a + a.T)

    # Add small jitter to diagonal for conditioning
    a = a + eps * torch.eye(B, device=a.device, dtype=a.dtype)

    # Eigen-decomposition (ascending eigenvalues)
    evals, evecs = torch.linalg.eigh(a)

    # Sort descending
    idx = torch.argsort(evals, descending=True)
    evals = evals[idx]
    evecs = evecs[:, idx]

    # Return top components
    h = evecs[:, :h_dim]
    s = evals  # (B,)

    return h, s

In [6]:
model_1V = torch.load('./out/Final-1V.tar', weights_only=False)
model_2V_TS = torch.load('./out/Final-2V-TS.tar', weights_only=False)
model_2V_TL = torch.load('./out/Final-2V-TL.tar', weights_only=False)
model_3V = torch.load('./out/Final-3V.tar', weights_only=False)

In [7]:
# Single View Networks
net_text_en_1V = model_1V['net_text_en']
net_text_de_1V = model_1V['net_text_de']

# Two View Networks - Image + Sentiment
net_text_en_2V_TS = model_2V_TS['net_text_en']
net_sent_en_2V_TS = model_2V_TS['net_sent_en']

net_text_de_2V_TS = model_2V_TS['net_text_de']
net_sent_de_2V_TS = model_2V_TS['net_sent_de']

# Two View Networks - Image + Length
net_text_en_2V_TL = model_2V_TL['net_text_en']
net_len_en_2V_TL = model_2V_TL['net_len_en']

net_text_de_2V_TL = model_2V_TL['net_text_de']
net_len_de_2V_TL = model_2V_TL['net_len_de']

# Three View Networks - Image + Sentiment + Length
net_text_en_3V = model_3V['net_text_en']
net_sent_en_3V = model_3V['net_sent_en']
net_len_en_3V = model_3V['net_len_en']

net_text_de_3V = model_3V['net_text_de']
net_sent_de_3V = model_3V['net_sent_de']
net_len_de_3V = model_3V['net_len_de']

In [8]:
net_text_en_1V.eval()
net_text_en_2V_TS.eval()
net_sent_en_2V_TS.eval()
net_text_en_2V_TL.eval()
net_len_en_2V_TL.eval()
net_text_en_3V.eval()
net_sent_en_3V.eval()
net_len_en_3V.eval()

net_text_de_1V.eval()
net_text_de_2V_TS.eval()
net_sent_de_2V_TS.eval()
net_text_de_2V_TL.eval()
net_len_de_2V_TL.eval()
net_text_de_3V.eval()
net_sent_de_3V.eval()
net_len_de_3V.eval()

NetLenDe(
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=15, bias=True)
)

In [9]:
all_data_text, all_data_sent, all_data_len = [], [], []

text_en_1V = []
text_en_2V_TS, sent_en_2V_TS = [], []
text_en_2V_TL, len_en_2V_TL = [], []
text_en_3V, sent_en_3V, len_en_3V = [], [], []

text_de_1V = []
text_de_2V_IS, sent_de_2V_IS = [], []
text_de_2V_IL, len_de_2V_IL = [], []
text_de_3V, sent_de_3V, len_de_3V = [], [], []

text_loss_1V = []
text_loss_2V_TS, sent_loss_2V_TS, loss_2V_TS = [], [], []
text_loss_2V_TL, len_loss_2V_TL, loss_2V_TL = [], [], []
text_loss_3V, sent_loss_3V, len_loss_3V, loss_3V = [], [], [], []

U_1Vs, U_2V_TSs, U_2V_TLs, U_3Vs = [], [], [], []
V_2V_TSs, V_3Vs = [], []
W_2V_TLs, W_3Vs = [], []

adv_text_loss_1V = []
adv_text_loss_2V_TS, adv_sent_loss_2V_TS, adv_loss_2V_TS = [], [], []
adv_text_loss_2V_TL, adv_len_loss_2V_TL, adv_loss_2V_TL = [], [], []
adv_text_loss_3V, adv_sent_loss_3V, adv_len_loss_3V, adv_loss_3V = [], [], [], []

In [10]:
recon_loss1 = torch.nn.CrossEntropyLoss(ignore_index=PAD)
recon_loss2 = torch.nn.MSELoss()
recon_loss3 = torch.nn.MSELoss()

In [11]:
with torch.no_grad():
    for i, (text, sent, length) in enumerate(xtest):
        if i > i_val * 10:
            break
        
        text = text.to(opt.device)
        sent = sent.to(opt.device)
        length = length.to(opt.device)

        all_data_text.append(text.cpu())
        all_data_sent.append(sent.cpu())
        all_data_len.append(length.cpu())

        # Single View
        text_en_1V.append(net_text_en_1V(text).cpu())

        # Two View - Text + Sentiment
        text_en_2V_TS.append(net_text_en_2V_TS(text).cpu())
        sent_en_2V_TS.append(net_sent_en_2V_TS(sent).cpu())

        # Two View - Text + Length
        text_en_2V_TL.append(net_text_en_2V_TL(text).cpu())
        len_en_2V_TL.append(net_len_en_2V_TL(length).cpu())

        # Three View - Image + Sketch + Label
        text_en_3V.append(net_text_en_3V(text).cpu())
        sent_en_3V.append(net_sent_en_3V(sent).cpu())
        len_en_3V.append(net_len_en_3V(length).cpu())

In [12]:
# Convert lists of batch tensors to one tensor
all_data_text = torch.cat(all_data_text, dim=0)
all_data_sent = torch.cat(all_data_sent, dim=0)
all_data_len  = torch.cat(all_data_len, dim=0)

text_en_1V = torch.cat(text_en_1V, dim=0)

text_en_2V_TS = torch.cat(text_en_2V_TS, dim=0)
sent_en_2V_TS = torch.cat(sent_en_2V_TS, dim=0)

text_en_2V_TL = torch.cat(text_en_2V_TL, dim=0)
len_en_2V_TL  = torch.cat(len_en_2V_TL, dim=0)

text_en_3V = torch.cat(text_en_3V, dim=0)
sent_en_3V = torch.cat(sent_en_3V, dim=0)
len_en_3V  = torch.cat(len_en_3V, dim=0)

In [13]:
with torch.no_grad():
    for i in range(i_val):
        start, stop = i * 100, (i + 1) * 100

        # Get latent representations (current batch of 100 samples)
        h_1V, _ = kPCA(text_en_1V[start:stop], None, None, nviews=1)
        h_2V_TS, _ = kPCA(text_en_2V_TS[start:stop], sent_en_2V_TS[start:stop], None, nviews=2)
        h_2V_TL, _ = kPCA(text_en_2V_TL[start:stop], len_en_2V_TL[start:stop], None, nviews=2)
        h_3V, _ = kPCA(text_en_3V[start:stop], sent_en_3V[start:stop], len_en_3V[start:stop], nviews=3)

        # Calculate U // All views
        U_1V = text_en_1V[start:stop].T @ h_1V
        U_2V_TS = text_en_2V_TS[start:stop].T @ h_2V_TS
        U_2V_TL = text_en_2V_TL[start:stop].T @ h_2V_TL
        U_3V = text_en_3V[start:stop].T @ h_3V

        # Calculate V // 2V-IS and 3V only
        V_2V_TS = sent_en_2V_TS[start:stop].T @ h_2V_TS
        V_3V = sent_en_3V[start:stop].T @ h_3V

        # Calculate W // 2V-IL and 3V only
        W_2V_TL = len_en_2V_TL[start:stop].T @ h_2V_TL
        W_3V = len_en_3V[start:stop].T @ h_3V

        # # Save U, V, W to use on the adversarial samples later
        U_1Vs.append(U_1V.cpu())
        U_2V_TSs.append(U_2V_TS.cpu())
        U_2V_TLs.append(U_2V_TL.cpu())
        U_3Vs.append(U_3V.cpu())

        V_2V_TSs.append(V_2V_TS.cpu())
        V_3Vs.append(V_3V.cpu())

        W_2V_TLs.append(W_2V_TL.cpu())
        W_3Vs.append(W_3V.cpu())

        # Reconstruct images, sketches, labels from latent representations
        text_1V_tilde, _ = net_text_de_1V(h_1V @ U_1V.T)
        text_2V_TS_tilde, _ = net_text_de_2V_TS(h_2V_TS @ U_2V_TS.T)
        text_2V_TL_tilde, _ = net_text_de_2V_TL(h_2V_TL @ U_2V_TL.T)
        text_3V_tilde, _ = net_text_de_3V(h_3V @ U_3V.T)

        sent_2V_TS_tilde = net_sent_de_2V_TS(h_2V_TS @ V_2V_TS.T)
        sent_3V_tilde = net_sent_de_3V(h_3V @ V_3V.T)

        len_2V_TL_tilde = net_len_de_2V_TL(h_2V_TL @ W_2V_TL.T)
        len_3V_tilde = net_len_de_3V(h_3V @ W_3V.T)

        # Calculate losses
        text_true = all_data_text[start:stop]
        sent_true = all_data_sent[start:stop]
        len_true = all_data_len[start:stop]

        # x_tilde.reshape(-1, x_tilde.size(-1)), X.reshape(-1).long()

        text_loss_1V.append(recon_loss1(text_1V_tilde.reshape(-1, text_1V_tilde.size(-1)), text_true.reshape(-1).long()).item())
        text_loss_2V_TS.append(recon_loss1(text_2V_TS_tilde.reshape(-1, text_2V_TS_tilde.size(-1)), text_true.reshape(-1).long()).item())
        text_loss_2V_TL.append(recon_loss1(text_2V_TL_tilde.reshape(-1, text_2V_TL_tilde.size(-1)), text_true.reshape(-1).long()).item())
        text_loss_3V.append(recon_loss1(text_3V_tilde.reshape(-1, text_3V_tilde.size(-1)), text_true.reshape(-1).long()).item())

        sent_loss_2V_TS.append(recon_loss2(sent_2V_TS_tilde, sent_true).item())
        sent_loss_3V.append(recon_loss2(sent_3V_tilde, sent_true).item())

        len_loss_2V_TL.append(recon_loss3(len_2V_TL_tilde, len_true).item())
        len_loss_3V.append(recon_loss3(len_3V_tilde, len_true).item())

        # Store some samples for visualization
        if i == 0:
            text_de_1V = text_1V_tilde.cpu()[:,:10]
            text_de_2V_TS = text_2V_TS_tilde.cpu()[:,:10]
            text_de_2V_TL = text_2V_TL_tilde.cpu()[:,:10]
            text_de_3V = text_3V_tilde.cpu()[:,:10]

            sent_de_2V_TS = sent_2V_TS_tilde.cpu()[:10]
            sent_de_3V = sent_3V_tilde.cpu()[:10]
            
            len_de_2V_TL = len_2V_TL_tilde.cpu()[:10]
            len_de_3V = len_3V_tilde.cpu()[:10]

        print(f'Batch {i + 1}/{i_val} processed.')

Batch 1/6 processed.
Batch 2/6 processed.
Batch 3/6 processed.
Batch 4/6 processed.
Batch 5/6 processed.
Batch 6/6 processed.


## Attack

In [14]:
epsilon = 0.2

data_text = all_data_text.cpu()
data_sent = all_data_sent.cpu()
data_len = all_data_len.cpu()

adv_text = []

In [15]:
augmenter = O200SemanticTokenAugmenter(
    max_len=15,
    pad_token=PAD,
    n_min=1,
    n_max=2,
    seed=42
)

In [16]:
for i in range(i_val):
    start, stop = i * 100, (i + 1) * 100

    batch = all_data_text[start:stop]
    adv_batch = augmenter.augment_batch(batch)

    adv_text.append(adv_batch.cpu())

In [17]:
print(f'---- Running Adversarial Batches through the Model ----')
adv_xs_tilde, adv_ys_tilde, adv_zs_tilde = [], [], []

with torch.no_grad():
    for i in range(i_val):
        print(f' > Processing adversarial batch {i + 1}/{i_val}...')
        start, stop = i * 100, (i + 1) * 100
        
        adv_text_batch = adv_text[i].to(opt.device)
        data_sent_batch = data_sent[start:stop].to(opt.device)
        data_len_batch = data_len[start:stop].to(opt.device)

        adv_out_text_1V = net_text_en_1V(adv_text_batch)
        adv_out_text_2V_TS = net_text_en_2V_TS(adv_text_batch)
        adv_out_text_2V_TL = net_text_en_2V_TL(adv_text_batch)
        adv_out_text_3V = net_text_en_3V(adv_text_batch)

        adv_out_sent_2V_TS = net_sent_en_2V_TS(data_sent_batch)
        adv_out_sent_3V = net_sent_en_3V(data_sent_batch)

        adv_out_len_2V_TL = net_len_en_2V_TL(data_len_batch)
        adv_out_len_3V = net_len_en_3V(data_len_batch)

        # Get latent representations (current batch of 100 samples)
        h_1V, _ = kPCA(adv_out_text_1V, None, None, nviews=1)
        h_2V_TS, _ = kPCA(adv_out_text_2V_TS, adv_out_sent_2V_TS, None, nviews=2)
        h_2V_TL, _ = kPCA(adv_out_text_2V_TL, adv_out_len_2V_TL, None, nviews=2)
        h_3V, _ = kPCA(adv_out_text_3V, adv_out_sent_3V, adv_out_len_3V, nviews=3)

        # Use Pre computed U, V, W from the clean samples (same batch index)
        U_1V = U_1Vs[i].to(opt.device)
        U_2V_TS = U_2V_TSs[i].to(opt.device)
        U_2V_TL = U_2V_TLs[i].to(opt.device)
        U_3V = U_3Vs[i].to(opt.device)

        V_2V_TS = V_2V_TSs[i].to(opt.device)
        V_3V = V_3Vs[i].to(opt.device)

        W_2V_TL = W_2V_TLs[i].to(opt.device)
        W_3V = W_3Vs[i].to(opt.device)

        # Reconstruct images, sketches, labels from latent representations
        text_1V_tilde, _ = net_text_de_1V(h_1V @ U_1V.T)
        text_2V_TS_tilde, _ = net_text_de_2V_TS(h_2V_TS @ U_2V_TS.T)
        text_2V_TL_tilde, _ = net_text_de_2V_TL(h_2V_TL @ U_2V_TL.T)
        text_3V_tilde, _ = net_text_de_3V(h_3V @ U_3V.T)

        sent_2V_TS_tilde = net_sent_de_2V_TS(h_2V_TS @ V_2V_TS.T)
        sent_3V_tilde = net_sent_de_3V(h_3V @ V_3V.T)

        len_2V_TL_tilde = net_len_de_2V_TL(h_2V_TL @ W_2V_TL.T)
        len_3V_tilde = net_len_de_3V(h_3V @ W_3V.T)

        # Calculate losses
        text_true = all_data_text[start:stop]
        sent_true = all_data_sent[start:stop]
        len_true = all_data_len[start:stop]

        # x_tilde.reshape(-1, x_tilde.size(-1)), X.reshape(-1).long()

        adv_text_loss_1V.append(recon_loss1(text_1V_tilde.reshape(-1, text_1V_tilde.size(-1)), text_true.reshape(-1).long()).item())
        adv_text_loss_2V_TS.append(recon_loss1(text_2V_TS_tilde.reshape(-1, text_2V_TS_tilde.size(-1)), text_true.reshape(-1).long()).item())
        adv_text_loss_2V_TL.append(recon_loss1(text_2V_TL_tilde.reshape(-1, text_2V_TL_tilde.size(-1)), text_true.reshape(-1).long()).item())
        adv_text_loss_3V.append(recon_loss1(text_3V_tilde.reshape(-1, text_3V_tilde.size(-1)), text_true.reshape(-1).long()).item())

        adv_sent_loss_2V_TS.append(recon_loss2(sent_2V_TS_tilde, sent_true).item())
        adv_sent_loss_3V.append(recon_loss2(sent_3V_tilde, sent_true).item())

        adv_len_loss_2V_TL.append(recon_loss3(len_2V_TL_tilde, len_true).item())
        adv_len_loss_3V.append(recon_loss3(len_3V_tilde, len_true).item())

---- Running Adversarial Batches through the Model ----
 > Processing adversarial batch 1/6...
 > Processing adversarial batch 2/6...
 > Processing adversarial batch 3/6...
 > Processing adversarial batch 4/6...
 > Processing adversarial batch 5/6...
 > Processing adversarial batch 6/6...


## Attack Loss

In [18]:
# # test for significance of differences using paired t-tests
# C_F_im_test = stats.ttest_rel(clean_im_loss, fgsm_losses_im)
# C_B_im_test = stats.ttest_rel(clean_im_loss, bim_losses_im)
# F_B_im_test = stats.ttest_rel(fgsm_losses_im, bim_losses_im)

# print(f'---- Paired t-test results for Image Loss: ----')
# print(f' > Clean vs FGSM: t-statistic = {C_F_im_test.statistic:.4f}, p-value = {C_F_im_test.pvalue}')
# print(f' > Clean vs BIM: t-statistic = {C_B_im_test.statistic:.4f}, p-value = {C_B_im_test.pvalue}')
# print(f' > FGSM vs BIM: t-statistic = {F_B_im_test.statistic:.4f}, p-value = {F_B_im_test.pvalue}')

In [19]:
models = ['1V', '2V-TS', '2V-TL', '3V']

clean_text_losses = [text_loss_1V, text_loss_2V_TS, text_loss_2V_TL, text_loss_3V]
adv_text_losses = [adv_text_loss_1V, adv_text_loss_2V_TS, adv_text_loss_2V_TL, adv_text_loss_3V]

clean_sent_losses = [None, sent_loss_2V_TS, None, sent_loss_3V]
adv_sent_losses = [None, adv_sent_loss_2V_TS, None, adv_sent_loss_3V]

clean_len_losses = [None, None, len_loss_2V_TL, len_loss_3V]
adv_len_losses = [None, None, adv_len_loss_2V_TL, adv_len_loss_3V]

In [21]:
for i, x in enumerate(models):
    r = pd.DataFrame({
        'model': [x] * i_val,

        'Clean_TEXT_Loss': clean_text_losses[i],
        'ADV_TEXT_Loss': adv_text_losses[i],
        'Clean_SENT_Loss': clean_sent_losses[i] if clean_sent_losses[i] is not None else [None] * i_val,
        'ADV_SENT_Loss': adv_sent_losses[i] if adv_sent_losses[i] is not None else [None] * i_val,
        'Clean_LEN_Loss': clean_len_losses[i] if clean_len_losses[i] is not None else [None] * i_val,
        'ADV_LEN_Loss': adv_len_losses[i] if adv_len_losses[i] is not None else [None] * i_val,
    })

    results_file = 'Attack-A-Results.csv'
    if os.path.exists(results_file):
        r.to_csv(results_file, mode='a', header=False, index=False)
    else:
        r.to_csv(results_file, index=False)